In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.optim as optim

LeNet-5 的设计思想是逐层提取特征：低层卷积捕捉基本边缘和纹理，中层卷积组合局部特征，高层
卷积和全连接层整合全局模式。池化层通过下采样提高平移不变性并降低计算量，全连接层将局部特征整
合为分类结果。这种结构在手写数字识别和小尺寸图像分类任务中表现优秀。

In [17]:
class LeNet5(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
    self.conv2 = nn.Conv2d(6, 16,kernel_size=5)
    self.conv3 = nn.Conv2d(16, 120, kernel_size=5)
    self.fc1 = nn.Linear(120, 84)
    self.fc2 = nn.Linear(84, 10)

  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = F.avg_pool2d(x,2)
    x = F.relu(self.conv2(x))
    x = F.avg_pool2d(x,2)
    x = F.relu(self.conv3(x))
    x = x.view(x.size(0), -1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

In [18]:
"""
transforms.ToTensor() —— 类型转换与归一化
把普通的 Python 图片对象（PIL Image）或 NumPy 数组转换成 PyTorch Tensor（张量），这样 GPU 才能处理。
初步归一化 (Scaling)：
原始图片的像素值通常是 0 到 255（0 是黑，255 是白）。
ToTensor() 会自动将这些值除以 255，把范围缩放到 [0.0, 1.0] 之间。
transforms.Normalize((0.5,), (0.5,)) —— 标准化
这一步通过公式进行平移和缩放：output = (input - mean) / std
参数含义：第一个 (0.5,) 是均值（Mean），第二个 (0.5,) 是标准差（Std）。因为 MNIST 是灰度图，所以只有一个通道。
"""
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [19]:
model = LeNet5()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epoch = 5
for epoch in range(num_epoch):
  model.train()
  running_loss = 0.0

  for batch_idx, (inputs, labels) in enumerate(train_loader):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    if batch_idx % 100 == 99:
      print(
          f"Epoch: {epoch + 1}/{num_epoch}, "
          f"Batch: {batch_idx + 1}/{len(train_loader)}, "
          f"Loss: {running_loss / 100:.4f}"
      )
      running_loss = 0.0
print("Finish training")
torch.save(model.state_dict(), 'lenet5.pth')

Epoch: 1/5, Batch: 100/938, Loss: 1.1114
Epoch: 1/5, Batch: 200/938, Loss: 0.3546
Epoch: 1/5, Batch: 300/938, Loss: 0.2863
Epoch: 1/5, Batch: 400/938, Loss: 0.2278
Epoch: 1/5, Batch: 500/938, Loss: 0.1667
Epoch: 1/5, Batch: 600/938, Loss: 0.1399
Epoch: 1/5, Batch: 700/938, Loss: 0.1418
Epoch: 1/5, Batch: 800/938, Loss: 0.1223
Epoch: 1/5, Batch: 900/938, Loss: 0.1084
Epoch: 2/5, Batch: 100/938, Loss: 0.0872
Epoch: 2/5, Batch: 200/938, Loss: 0.0840
Epoch: 2/5, Batch: 300/938, Loss: 0.0865
Epoch: 2/5, Batch: 400/938, Loss: 0.0763
Epoch: 2/5, Batch: 500/938, Loss: 0.0744
Epoch: 2/5, Batch: 600/938, Loss: 0.0729
Epoch: 2/5, Batch: 700/938, Loss: 0.0724
Epoch: 2/5, Batch: 800/938, Loss: 0.0646
Epoch: 2/5, Batch: 900/938, Loss: 0.0639
Epoch: 3/5, Batch: 100/938, Loss: 0.0631
Epoch: 3/5, Batch: 200/938, Loss: 0.0517
Epoch: 3/5, Batch: 300/938, Loss: 0.0520
Epoch: 3/5, Batch: 400/938, Loss: 0.0497
Epoch: 3/5, Batch: 500/938, Loss: 0.0572
Epoch: 3/5, Batch: 600/938, Loss: 0.0566
Epoch: 3/5, Batc

In [22]:
model = LeNet5()
model.load_state_dict(torch.load('lenet5.pth'))
model.eval()
for batch_idx,  (inputs, labels) in enumerate(test_loader):
  output_batch = model(inputs)
  predicted_classes = torch.argmax(output_batch, dim=1)
  print(f"Batch {batch_idx + 1}")
  print(f"Predicted Classes: {predicted_classes}")
  print(f"True Labels      : {labels}")
  print("-"*40)

Batch 1
Predicted Classes: tensor([7, 2, 1, 0, 4, 1, 4, 9, 5, 9, 0, 6, 9, 0, 1, 5, 9, 7, 3, 4, 9, 6, 6, 5,
        4, 0, 7, 4, 0, 1, 3, 1, 3, 4, 7, 2, 7, 1, 2, 1, 1, 7, 4, 2, 3, 5, 1, 2,
        4, 4, 6, 3, 5, 5, 6, 0, 4, 1, 9, 5, 7, 8, 9, 3])
True Labels      : tensor([7, 2, 1, 0, 4, 1, 4, 9, 5, 9, 0, 6, 9, 0, 1, 5, 9, 7, 3, 4, 9, 6, 6, 5,
        4, 0, 7, 4, 0, 1, 3, 1, 3, 4, 7, 2, 7, 1, 2, 1, 1, 7, 4, 2, 3, 5, 1, 2,
        4, 4, 6, 3, 5, 5, 6, 0, 4, 1, 9, 5, 7, 8, 9, 3])
----------------------------------------
Batch 2
Predicted Classes: tensor([7, 4, 6, 4, 3, 0, 7, 0, 2, 9, 1, 7, 3, 2, 9, 7, 7, 6, 2, 7, 8, 4, 7, 3,
        6, 1, 3, 6, 9, 3, 1, 4, 1, 7, 6, 9, 6, 0, 5, 4, 9, 9, 2, 1, 9, 4, 8, 7,
        3, 9, 7, 9, 4, 4, 9, 2, 5, 4, 7, 6, 7, 9, 0, 5])
True Labels      : tensor([7, 4, 6, 4, 3, 0, 7, 0, 2, 9, 1, 7, 3, 2, 9, 7, 7, 6, 2, 7, 8, 4, 7, 3,
        6, 1, 3, 6, 9, 3, 1, 4, 1, 7, 6, 9, 6, 0, 5, 4, 9, 9, 2, 1, 9, 4, 8, 7,
        3, 9, 7, 4, 4, 4, 9, 2, 5, 4, 7, 6, 7, 9, 0, 5])